# Proyecto 3 — Introducción a la Inteligencia Artificial
## Predicción de Alto Rendimiento Académico Universitario a partir de Datos Biométricos de Sueño (Fitbit)

**Dataset (Benchmark):** *CMU Sleep / Sleep vs. GPA in College* (Creswell et al., *PNAS* 2023)  
- **Kaggle:** https://www.kaggle.com/datasets/mexwell/cmu-sleep  
- **Repositorio Oficial CMU:** https://cmustatistics.github.io/data-repository/psychology/cmu-sleep.html

**Técnicas de Inteligencia Computacional Comparadas:**
1. Random Forest (**RF**)
2. Support Vector Machines (**SVM**)
3. Redes Neuronales — Perceptrón Multicapa (**MLP**)

**Diseño Experimental (Fase 3):**
- **Problema:** Clasificación binaria (`Alto_Rendimiento = 1` si `term_gpa >= 3.5`, `0` en caso contrario).
- **Variable Dependiente (VD):** `F1-Score Macro` (derivada de la matriz de confusión), apoyada en `Accuracy`, `Precision`, `Recall` y `ROC-AUC`.
- **Variables Independientes (VI):** 3 hiperparámetros por técnica × 3 niveles (*Bajo, Defecto, Alto*) = **9 casos por técnica** (27 evaluaciones con Validación Cruzada Estratificada 5-Fold).

## 1. Importación de Librerías y Carga del Dataset

In [ ]:
import os
import warnings
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, precision_score, recall_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

SEED = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
UMBRAL_GPA = 3.5

# Carga automática (funciona en local y en Google Colab)
os.makedirs('data', exist_ok=True)
CSV_PATH = 'data/cmu-sleep.csv'
if not os.path.exists(CSV_PATH):
    urllib.request.urlretrieve('https://cmustatistics.github.io/data-repository/data/cmu-sleep.csv', CSV_PATH)

df_raw = pd.read_csv(CSV_PATH)
print(f'Dimensiones originales: {df_raw.shape}')
df_raw.head()

## 2. Fase 2 — Diagnóstico, Limpieza, Preparación y Análisis Exploratorio (EDA)
En la inspección real del dataset se identifican hallazgos clave de calidad de datos:
1. **Valores faltantes ocultos:** En lugar de `NaN`, existen celdas con espacios en blanco (`' '`) en `demo_race` (1), `demo_gender` (3), `demo_firstgen` (4), `term_units` (147) y `Zterm_units_ZofZ` (147 en la cohorte `nh`).
2. **Anomalía de dominio:** `demo_firstgen` es una variable binaria `{0, 1}`, pero contiene 1 registro corrupto con valor `2`.
3. **Escalas heterogéneas:** `term_units` usa distintas escalas entre universidades (CMU vs UW), por lo que se descarta y se usa `Zterm_units_ZofZ` (estandarizada por cohorte).
4. **Asimetría extrema:** `bedtime_mssd` tiene asimetría > 9.0, corregida mediante `log1p(bedtime_mssd)`.
5. **Cero fuga de datos (*data leakage*):** `cum_gpa` es el promedio *anterior* al semestre estudiado; además, la imputación (`SimpleImputer`) y normalización (`StandardScaler`) se ejecutan estrictamente dentro del `Pipeline` en cada fold de CV.

In [ ]:
# 1. Re-lectura convirtiendo espacios en blanco en NaN
df = pd.read_csv(CSV_PATH, na_values=[' ', ''])
print('Valores faltantes reales detectados:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2. Corregir valor inválido de dominio en demo_firstgen (2 -> NaN)
df['demo_firstgen'] = df['demo_firstgen'].replace(2, np.nan)

# 3. Transformación logarítmica para reducir asimetría en variabilidad de hora de dormir
print(f"Asimetría bedtime_mssd antes: {df['bedtime_mssd'].skew():.2f}")
df['log_bedtime_mssd'] = np.log1p(df['bedtime_mssd'])
print(f"Asimetría log_bedtime_mssd después: {df['log_bedtime_mssd'].skew():.2f}")

# 4. Variable objetivo binaria: Alto Rendimiento (term_gpa >= 3.5)
df['Alto_Rendimiento'] = (df['term_gpa'] >= UMBRAL_GPA).astype(int)
print('\nDistribución de Clases:')
print(df['Alto_Rendimiento'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

num_features = [
    'TotalSleepTime', 'log_bedtime_mssd', 'midpoint_sleep', 'daytime_sleep',
    'frac_nights_with_data', 'cum_gpa', 'Zterm_units_ZofZ'
]
cat_bin_features = ['demo_race', 'demo_gender', 'demo_firstgen']
X = df[num_features + cat_bin_features].copy()
y = df['Alto_Rendimiento'].copy()

In [ ]:
# Visualización Exploratoria (EDA)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.histplot(df['term_gpa'], bins=25, kde=True, color='#2563EB', ax=axes[0])
axes[0].axvline(UMBRAL_GPA, color='#DC2626', linestyle='--', lw=2, label=f'Umbral = {UMBRAL_GPA}')
axes[0].set_title('Distribución de GPA Semestral')
axes[0].legend()

df_plot = df.assign(Horas_Sueno=df['TotalSleepTime'] / 60.0,
                    Clase=df['Alto_Rendimiento'].map({0: 'Estándar/Riesgo (<3.5)', 1: 'Alto Rendimiento (>=3.5)'}))
sns.boxplot(data=df_plot, x='Clase', y='Horas_Sueno', palette=['#F59E0B', '#10B981'], ax=axes[1])
axes[1].set_title('Horas de Sueño (Fitbit) por Clase')
axes[1].set_xlabel('')

corr_cols = ['term_gpa', 'cum_gpa', 'TotalSleepTime', 'midpoint_sleep', 'daytime_sleep', 'log_bedtime_mssd', 'Zterm_units_ZofZ']
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, ax=axes[2])
axes[2].set_title('Matriz de Correlación')
plt.tight_layout()
plt.show()

## 3. Pipeline de Preprocesamiento y Evaluación (Stratified 5-Fold CV)

In [ ]:
def construir_pipeline(modelo, escalar=True):
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if escalar:
        num_steps.append(('scaler', StandardScaler()))
    prep = ColumnTransformer([
        ('num', Pipeline(num_steps), num_features),
        ('cat', SimpleImputer(strategy='most_frequent'), cat_bin_features),
    ])
    return Pipeline([('prep', prep), ('clf', modelo)])

def evaluar(pipe):
    res = cross_validate(pipe, X, y, cv=CV, scoring=['f1_macro', 'accuracy', 'precision_macro', 'recall_macro', 'roc_auc'], n_jobs=-1)
    return {
        'F1_Macro': round(res['test_f1_macro'].mean(), 4),
        'F1_Std': round(res['test_f1_macro'].std(), 4),
        'Accuracy': round(res['test_accuracy'].mean(), 4),
        'Precision': round(res['test_precision_macro'].mean(), 4),
        'Recall': round(res['test_recall_macro'].mean(), 4),
        'ROC_AUC': round(res['test_roc_auc'].mean(), 4),
    }

## 4. Fase 3 — Diseño Experimental Oficial (3 Hiperparámetros × 3 Niveles = 9 Casos por Técnica)
Tras pruebas preliminares de sensibilidad, se seleccionaron los 3 hiperparámetros con mayor impacto sobre `F1-Score Macro` para cada técnica (RF, SVM y MLP), comparando un valor **Bajo**, el valor **Defecto/Base** y un valor **Alto**.

In [ ]:
experimentos = {
    'Random Forest': {
        'escalar': False,
        'base': {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 5, 'random_state': SEED},
        'cls': RandomForestClassifier,
        'factores': {
            'n_estimators': [('Bajo', 5), ('Defecto', 100), ('Alto', 500)],
            'max_depth': [('Bajo', 1), ('Defecto', 10), ('Alto', 30)],
            'min_samples_leaf': [('Bajo', 1), ('Defecto', 5), ('Alto', 80)],
        },
    },
    'SVM': {
        'escalar': True,
        'base': {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf', 'probability': True, 'random_state': SEED},
        'cls': SVC,
        'factores': {
            'C': [('Bajo', 0.01), ('Defecto', 1.0), ('Alto', 100.0)],
            'gamma': [('Bajo', 0.001), ('Defecto', 'scale'), ('Alto', 2.0)],
            'kernel': [('Bajo (Lineal)', 'linear'), ('Defecto (RBF)', 'rbf'), ('Alto (Poli-3)', 'poly')],
        },
    },
    'MLP': {
        'escalar': True,
        'base': {'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001, 'max_iter': 200, 'random_state': SEED},
        'cls': MLPClassifier,
        'factores': {
            'hidden_layer_sizes': [('Bajo', (3,)), ('Defecto', (100,)), ('Alto', (300, 150))],
            'learning_rate_init': [('Bajo', 0.0001), ('Defecto', 0.001), ('Alto', 0.5)],
            'max_iter': [('Bajo', 20), ('Defecto', 200), ('Alto', 1000)],
        },
    },
}

tablas = {}
mejores = {}

for tec, cfg in experimentos.items():
    filas, best_f1, best_obj = [], -1, None
    c = 1
    for hp, niveles in cfg['factores'].items():
        for niv, val in niveles:
            p = dict(cfg['base']); p[hp] = val
            m = evaluar(construir_pipeline(cfg['cls'](**p), escalar=cfg['escalar']))
            filas.append({'Caso': f'C{c}', 'Hiperparametro': hp, 'Nivel': niv, 'Valor': str(val), **m})
            if m['F1_Macro'] > best_f1:
                best_f1 = m['F1_Macro']
                best_obj = {'Tecnica': tec, 'Caso': f'C{c}', 'Config': f'{hp}={val}', 'Params': p, 'Escalar': cfg['escalar'], 'Cls': cfg['cls'], **m}
            c += 1
    tablas[tec] = pd.DataFrame(filas)
    mejores[tec] = best_obj
    print(f'\n=== RESULTADOS EXPERIMENTO: {tec} (9 Casos) ===')
    display(tablas[tec])

## 5. Gráficas de Resultados por Técnica y Comparativa con Matrices de Confusión

In [ ]:
# 1. Gráficas individuales por técnica
for tec, df_t in tablas.items():
    hps = df_t['Hiperparametro'].unique()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
    for i, hp in enumerate(hps):
        sub = df_t[df_t['Hiperparametro'] == hp]
        lbls = [f'{n}\n{v}' if str(v).startswith('(') else f'{n}\n({v})' for n, v in zip(sub['Nivel'], sub['Valor'])]
        bars = axes[i].bar(lbls, sub['F1_Macro'], yerr=sub['F1_Std'], capsize=5, color=['#64748B', '#2563EB', '#0EA5E9'])
        bars[sub['F1_Macro'].values.argmax()].set_color('#10B981')
        for b, val, sd in zip(bars, sub['F1_Macro'], sub['F1_Std']):
            axes[i].text(b.get_x() + b.get_width()/2, val + sd + 0.014, f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
        axes[i].set_title(hp, fontweight='bold')
        axes[i].set_ylim(0.25, 0.90)
        if i == 0:
            axes[i].set_ylabel('F1-Score Macro (5-Fold CV)')
    fig.suptitle(f'Experimento (9 Casos) — {tec}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 2. Matrices de confusión (Out-of-Fold) y Tabla Comparativa Final
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
resumen_final = []
for idx, (tec, info) in enumerate(mejores.items()):
    pipe = construir_pipeline(info['Cls'](**info['Params']), escalar=info['Escalar'])
    y_pred = cross_val_predict(pipe, X, y, cv=CV, n_jobs=-1)
    cm = confusion_matrix(y, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=['Blues', 'Greens', 'Purples'][idx], cbar=False,
                xticklabels=['Estándar (<3.5)', 'Alto (>=3.5)'], yticklabels=['Estándar (<3.5)', 'Alto (>=3.5)'], ax=axes[idx])
    axes[idx].set_title(f"{tec}\n({info['Config']}) | F1={info['F1_Macro']:.3f}", fontweight='bold')
    axes[idx].set_xlabel('Predicción')
    if idx == 0:
        axes[idx].set_ylabel('Real')
    resumen_final.append({
        'Tecnica': tec, 'Mejor_Caso': info['Caso'], 'Configuracion': info['Config'],
        'F1_Macro': info['F1_Macro'], 'Accuracy': info['Accuracy'], 'ROC_AUC': info['ROC_AUC'],
        'TN': cm[0,0], 'FP': cm[0,1], 'FN': cm[1,0], 'TP': cm[1,1]
    })
plt.tight_layout()
plt.show()

df_final = pd.DataFrame(resumen_final).sort_values('F1_Macro', ascending=False)
display(df_final)